In [2]:
# Init
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv(override=True)

api_key = os.getenv("API_KEY")
base_url = os.getenv("BASE_URL")
model = os.getenv("MODEL")

client = Anthropic(api_key=api_key, base_url=base_url)

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, schema=None):
    params = {
        "model": model,
        "max_tokens": 5000,
        "messages": messages
    }

    if system:
        params["system"] = system
    if schema:
        params["output_config"] = {"format": {"type": "json_schema", "schema": schema}}

    message = client.messages.create(**params)
    #return message.content[0].text
    #
    # a fix for ThinkingBlock error
    return "".join(b.text for b in message.content if b.type == "text")

In [3]:
import json

dataset_schema = {
  "type": "array",
  "items": {
    "type": "object",
    "properties": {
      "task": {"type": "string", "description": "Description of task"},
      "format": {"type": "string", "enum": ["json", "python", "regex"]}
    },
    "required": ["task", "format"],
    "additionalProperties": False,
  },
}

def generate_dataset():
  prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

  messages = []
  add_user_message(messages, prompt)
  text = chat(messages, schema=dataset_schema)
  return json.loads(text)


generate_dataset()

[{'task': 'Write a Python function using boto3 that takes a bucket name and returns the number of objects stored in that S3 bucket, handling pagination with a paginator.',
  'format': 'python'},
 {'task': "Write a JSON IAM policy document that allows read-only access (s3:GetObject and s3:ListBucket) to the S3 bucket named 'reports-archive' and nothing else.",
  'format': 'json'},
 {'task': "Write a regex that matches valid AWS EC2 instance IDs, which begin with 'i-' followed by either 8 or 17 lowercase hexadecimal characters.",
  'format': 'regex'}]

In [4]:
dataset = generate_dataset()

# Save dataset to file
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [5]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with JSON, Python, or a plain RegEx
* Do not add any comments, or commentary, or explanation
* Return an actual runnable code only, do not wrap it in markdown backticks
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [6]:
grade_schema = {
    "type": "object",
    "properties": {
        "strengths": {
            "type": "array",
            "items": {"type": "string"},
            "description": "1-3 key strengths",
        },
        "weaknesses": {
            "type": "array",
            "items": {"type": "string"},
            "description": "1-3 key areas for improvement",
        },
        "reasoning": {
            "type": "string",
            "description": "A concise explanation of your assessment",
        },
        "score": {
            "type": "integer",
            "description": "A number between 1 and 10",
        },
    },
    "required": ["strengths", "weaknesses", "reasoning", "score"],
    "additionalProperties": False,
}

def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)

    eval_text = chat(messages, schema=grade_schema)
    return json.loads(eval_text)

In [7]:
import ast
import re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    if format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)
    

In [8]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    code_score = grade_syntax(output, test_case)
    print(test_case["format"], "model_score", model_score, "code_score", code_score)

    score = (model_score + code_score) / 2

    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print("Average score:", average_score)
    
    return results

In [9]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

result = run_eval(dataset)

print(json.dumps(result, indent=2))

python model_score 8 code_score 10
json model_score 9 code_score 10
regex model_score 9 code_score 10
Average score: 9.333333333333334
[
  {
    "output": "import boto3\n\ndef count_s3_objects(bucket_name):\n    s3 = boto3.client('s3')\n    paginator = s3.get_paginator('list_objects_v2')\n    count = 0\n    for page in paginator.paginate(Bucket=bucket_name):\n        count += page.get('KeyCount', 0)\n    return count",
    "test_case": {
      "task": "Write a Python function using boto3 that takes a bucket name and returns the number of objects stored in the S3 bucket, handling pagination.",
      "format": "python"
    },
    "score": 9.0,
    "reasoning": "The solution correctly addresses the core requirement: it uses a paginator so that all objects are counted even when the bucket holds more than 1000 keys. KeyCount on each page of list_objects_v2 accurately reflects the number of keys returned in that page, so summing it yields the total object count. The code is short and idiomat